# Laya V3.3 — Exact Retrieval Oracle + Gaussian-Latent VAE Mathematical Audit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V33_Retrieval_Oracle_Gaussian_VAE_Audit_Colab.ipynb)

## Research questions

**Q1.** Does frozen pretrained ModernBERT **full-attention layer 15** require a few specific distant keys or a distributed set of keys? Compare exact native output to a 256-token local window, the teacher's *oracle* top-8/16/32/64 selected keys per query, and local-256 plus oracle remote top-8/16/32. Quantify captured teacher attention mass, teacher-output cosine and NMSE. **Oracle top-k uses the full teacher attention matrix; it is NOT an efficient attention replacement.**

**Q2.** Can compressing keys into 3-per-16 block landmarks preserve the teacher? Test masked mean, actual max-key-norm token, and a central real token, retaining matched key/value vectors and the pretrained Q/K RoPE. This is a deterministic diagnostic, **not an exact reproduction of V3.2's learned soft-salience landmark**.

**Q3.** Does placing K and V into a Gaussian latent space improve information preservation? Train **matched deterministic AEs, denoising AEs and β-VAEs** using the same frozen layer-15 K/V tensors. Evaluate β ∈ {0.0001, 0.001, 0.01, 0.1, 1.0} versus deterministic/noisy baselines at latent dimensions **16 and 32** (edit LATENT_DIMS for 64). The VAE uses a learned mean/log-variance encoder, reparameterization, and **KL to N(0,I)**. Report reconstructed K/V errors, attention output NMSE/cosine, aggregate Gaussianity diagnostics and stochastic reconstruction, not just KL loss.

**Q4.** For the best probe-selected AE and VAE, does compress-and-decode preserve downstream decision agreement, gold accuracy, and latency? This *diagnostic* keeps full QK SDPA after KV decoding: it is **not an attention-free or faster architecture**. If successful, a separate efficient retrieval design is still needed.

## Controls and interpretation

- Teacher: \`convaiinnovations/laya-typed-decisions\`, target layer **15**, full pretrained RoPE + original attention mask, no source fine-tuning.
- Case-disjoint train, probe, audit drawn **only from the dataset TRAIN split**; official test remains untouched. Train AE/VAEs only on training K/V; choose configurations by probe metrics; evaluate audit without tuning on it.
- Explicit teacher reconstruction must pass. Valid-query and valid-key masking applies throughout. Cache and sample sizes are printed.
- **Window 256** here means 127 positions to the left, query itself, 128 right: 256 possible key positions away from boundaries.
- \`top_k\` and local-plus-top-k are diagnostic upper-performance *construction examples*, not guaranteed optimal compression or bounds for every method. Per-head oracle picks are performed **after observing true teacher scores**.
- VAE may make the aggregated representation more nearly Gaussian; KL alone cannot guarantee it, and **independent Gaussian noise cannot recreate information already discarded**.
- We do not claim any model passes until Colab GPU output is produced. Stop further expensive training if reconstruction or locality diagnostics fail.


In [ ]:
#@title 1. Install exact training dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2033
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 400 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 64 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 8 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 8 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V33_Retrieval_Gaussian_VAE")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4 and TRAIN_CACHE_BATCHES > 0
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
#@title VAE and retrieval experiment controls
LATENT_DIMS = (16,32)  # Try (16,32,64) for the extended sweep
VAE_BETAS = (0.0001, 0.001, 0.01, 0.1, 1.0)
VAE_STEPS = 120 #@param {type:"integer"}
VAE_BATCH = 1024 #@param {type:"integer"}
VAE_TRAIN_SAMPLE_PER_BATCH = 1024 #@param {type:"integer"}
AUDIT_ALL_VAEs = True #@param {type:"boolean"}
RETRIEVAL_BATCHES = min(PROBE_CACHE_BATCHES, 4)
assert VAE_STEPS >= 1 and VAE_BATCH>=32 and VAE_TRAIN_SAMPLE_PER_BATCH>=32
print("Retrieval oracle + VAE training; official test used: NO")
print("VAE dimensions:",LATENT_DIMS,"betas:",VAE_BETAS,"steps/config:",VAE_STEPS)


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_embeddings=entry["pos"],
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V2.9 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


## 1. Mathematical diagnostics: original Q/K/V and oracle sparsity

At a query \(i\), native attention is \(o_i=W_O\sum_j \alpha_{ij} v_j\). For a selected subset \(S_i\), the renormalized sparse oracle is

\[
\widehat o_i=W_O\left(\frac{\sum_{j\in S_i}\alpha_{ij}v_j}
{\sum_{j\in S_i}\alpha_{ij}}\right).
\]

Its captured attention mass \(\sum_{j\in S_i}\alpha_{ij}\) measures coverage, but output error depends on the *values* and output projection. Small captured mass can still produce accurate outputs if values cancel; high coverage need not guarantee exact decisions. The full teacher's attention scores determine \(S_i\), so oracle timing is intentionally excluded from efficiency claims.

For the local-window experiment \(S_i=\{j: i-127\le j\le i+128\}\), intersected with valid keys. For \`local256 + topk_remote\`, the local indices plus teacher-selected remote indices are retained and renormalized.

For 3-per-16 landmarks, the query is compared against compressed mean and two actual-position key/value landmarks. This is **not** a theoretical upper bound on learned landmark models. Both key averaging after RoPE and compression can lose position/content identity.

Teacher reconstruction and each oracle are scored on exactly the same frozen teacher input/output tensors.


In [ ]:
#@title 4. Reproduce exact pretrained SDPA Q/K/V and attention weights
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

def manual_teacher_attention(entry):
    module = teacher.encoder.layers[LAYER].attn
    impl = module.config._attn_implementation
    if impl not in ("sdpa", "eager"):
        raise RuntimeError(f"Manual attention audit requires SDPA/eager, got {impl}")
    x = entry["x"]
    b, t, hidden = x.shape
    h, d = module.num_heads, module.head_dim
    kw = entry["call_kwargs"]
    position_ids = kw.get("position_ids")
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        qkv = module.Wqkv(x).view(b, t, 3, h, d)
        cos, sin = module.rotary_emb(qkv, position_ids=position_ids)
        q, k, v = qkv.transpose(3, 1).unbind(dim=2)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        mask = kw.get("attention_mask")
        if module.local_attention != (-1, -1):
            raise RuntimeError("Mathematical audit target is not a full-attention layer")
        core = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=0.0)
        exact_reconstruction = module.out_drop(
            module.Wo(core.transpose(1, 2).contiguous().view(b, t, hidden))
        )
    valid = entry["valid"]
    error = exact_reconstruction.float()[valid] - entry["y"].float()[valid]
    truth = entry["y"].float()[valid]
    reconstruction_nmse = float(error.square().mean()/truth.square().mean().clamp_min(1e-8))
    reconstruction_cos = float(F.cosine_similarity(
        exact_reconstruction.float()[valid], entry["y"].float()[valid], dim=-1
    ).mean())
    if reconstruction_nmse > .02:
        raise RuntimeError(
            f"Attention reconstruction mismatched model output (NMSE={reconstruction_nmse:.5f}). "
            "Do not interpret the attention statistics before fixing RoPE/masks."
        )

    # Explicit FP32 attention probabilities are for *diagnostics only*.
    # They are never computed by a CeNN replacement at inference.
    scores = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        if mask.dtype == torch.bool:
            scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        else:
            scores = scores + mask.float()
    weights = torch.softmax(scores, dim=-1).nan_to_num(0.0)
    valid_keys = valid[:, None, None, :].to(weights.dtype)
    weights = weights * valid_keys
    weights = weights / weights.sum(-1, keepdim=True).clamp_min(1e-8)
    return dict(q=q.detach(), k=k.detach(), v=v.detach(),
                weights=weights.detach(), valid=valid,
                reconstruction_nmse=reconstruction_nmse,
                reconstruction_cos=reconstruction_cos)


In [ ]:
#@title 5. Matched exact 256-window and sparse top-K retrieval oracles
RETRIEVAL_TOP_K = (8,16,32,64)
REMOTE_TOP_K = (8,16,32)
WINDOW_LEFT = 127
WINDOW_RIGHT = 128
assert WINDOW_LEFT + 1 + WINDOW_RIGHT == 256

def truth_project_core(entry,core):
    """Use native pretrained output projection on a replacement head-wise core."""
    module=teacher.encoder.layers[LAYER].attn
    b,h,t,d=core.shape
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        out=module.out_drop(module.Wo(
            core.to(amp_dtype).transpose(1,2).contiguous().reshape(b,t,h*d)
        ))
    return out

def metrics_with_valid(pred,truth,valid):
    p=pred.float()[valid]
    y=truth.float()[valid]
    return dict(
        cosine=float(F.cosine_similarity(p,y,dim=-1).mean()),
        nmse=float((p-y).square().sum()/y.square().sum().clamp_min(1e-8)),
        tokens=int(valid.sum())
    )

def weighted_oracle(entry,qkv,keep,weights=None):
    A=qkv["weights"] if weights is None else weights
    valid=qkv["valid"]
    v=qkv["v"]
    sel=A*keep.to(A.dtype)
    mass=sel.sum(-1,keepdim=True)
    sel=sel/mass.clamp_min(1e-10)
    # FP32 values limit avoidable oracle numerical artifacts.
    core=sel @ v.float()
    pred=truth_project_core(entry,core)
    met=metrics_with_valid(pred,entry["y"],valid)
    valid_queries=valid[:,None,:].expand_as(mass.squeeze(-1))
    met["captured_attention_mass"]=float(mass.squeeze(-1)[valid_queries].mean())
    met["fraction_mass_ge_90pct"]=float((mass.squeeze(-1)[valid_queries]>=.90).float().mean())
    return met

def oracle_batch(entry,qkv):
    A=qkv["weights"]
    B,H,T,_=A.shape
    pos=torch.arange(T,device=A.device)
    delta=pos[None,:]-pos[:,None]
    near=((delta>=-WINDOW_LEFT)&(delta<=WINDOW_RIGHT))[None,None,:,:]
    valid_keys=qkv["valid"][:,None,None,:]
    near=near&valid_keys
    output={}
    output["local_256"]=weighted_oracle(entry,qkv,near)
    for k in RETRIEVAL_TOP_K:
        topval,idx=torch.topk(A,k=min(k,T),dim=-1)
        keep=torch.zeros_like(A,dtype=torch.bool).scatter_(-1,idx,True)
        output[f"top_{k}"]=weighted_oracle(entry,qkv,keep)
    far=A.masked_fill(near,0)
    for k in REMOTE_TOP_K:
        _values,idx=torch.topk(far,k=min(k,T),dim=-1)
        remote_sel=torch.zeros_like(A,dtype=torch.bool).scatter_(-1,idx,True)
        output[f"local256_plus_remote_{k}"]=weighted_oracle(
            entry,qkv,near|remote_sel
        )
    # Teacher selectivity diagnostics.
    valid_q=qkv["valid"][:,None,:].expand(B,H,T)
    probs=A.clamp_min(1e-12)
    entropy=-(A*probs.log()).sum(-1)
    neff=1/A.square().sum(-1).clamp_min(1e-12)
    output["teacher_selectivity"]=dict(
        effective_num_keys=float(neff[valid_q].mean()),
        entropy_equiv_num_keys=float(entropy[valid_q].exp().mean()),
        largest_key_probability=float(A.max(-1).values[valid_q].mean()),
        remote_mass_outside_local256=float(A.masked_fill(near,0).sum(-1)[valid_q].mean())
    )
    return output

def mean_metric(items):
    return {key:float(np.mean([r[key] for r in items])) for key in items[0]}

probe_retrieval_cache=cache_teacher_io(LAYER,probe_items,RETRIEVAL_BATCHES)
probe_retrieval=[]
for i,entry in enumerate(probe_retrieval_cache):
    qkv=manual_teacher_attention(entry)
    row=oracle_batch(entry,qkv)
    probe_retrieval.append(row)
    print(f"Probe retrieval batch {i}: native reconstruction cosine="
          f"{qkv['reconstruction_cos']:.8f} NMSE={qkv['reconstruction_nmse']:.8g}")
    del qkv
ORACLE_NAMES=("local_256",)+tuple(f"top_{k}" for k in RETRIEVAL_TOP_K)+tuple(
    f"local256_plus_remote_{k}" for k in REMOTE_TOP_K)
probe_oracles={name:mean_metric([r[name] for r in probe_retrieval]) for name in ORACLE_NAMES}
teacher_selectivity=mean_metric([r["teacher_selectivity"] for r in probe_retrieval])
print("\\nActual attention selectivity:",teacher_selectivity)
print("Oracle PROBE — upper-performance construction using full teacher scores:")
for name,r in probe_oracles.items():
    print(f" {name:<29} cosine={r['cosine']:.5f} NMSE={r['nmse']:.5f}"
          f" mass={r['captured_attention_mass']:.3f}")


In [ ]:
#@title 6. Three-per-16 untrained landmark diagnostic (no learned Q/K selection)
def baseline_landmark(entry,qkv,block=16):
    q,k,v=qkv["q"],qkv["k"],qkv["v"]
    valid=qkv["valid"]
    B,H,T,D=q.shape
    extra=(-T)%block
    G=(T+extra)//block
    # [B,G,block,H,D], grouping complete K/V vectors by position.
    kk=k.transpose(1,2)
    vv=v.transpose(1,2)
    if extra:
        kk=F.pad(kk,(0,0,0,0,0,extra))
        vv=F.pad(vv,(0,0,0,0,0,extra))
        mask=F.pad(valid,(0,extra),value=False)
    else: mask=valid
    kg=kk.reshape(B,G,block,H,D)
    vg=vv.reshape(B,G,block,H,D)
    mm=mask.reshape(B,G,block)
    keep=mm[:,:,:,None,None]
    count=mm.sum(2).clamp_min(1)[:,:,None,None].to(kg.dtype)
    avgk=(kg*keep).sum(2)/count
    avgv=(vg*keep).sum(2)/count
    norms=kg.float().square().sum(-1).mean(-1).masked_fill(~mm,-1e20)
    maxidx=norms.argmax(2)
    mididx=((mm.sum(2).clamp_min(1)-1)//2).long()
    def selected(index):
        idx=index[:,:,None,None,None].expand(B,G,1,H,D)
        return kg.gather(2,idx).squeeze(2),vg.gather(2,idx).squeeze(2)
    maxk,maxv=selected(maxidx)
    midk,midv=selected(mididx)
    km=torch.stack((avgk,maxk,midk),dim=2).reshape(B,G*3,H,D).transpose(1,2)
    vm=torch.stack((avgv,maxv,midv),dim=2).reshape(B,G*3,H,D).transpose(1,2)
    allowed=mm.any(2).repeat_interleave(3,dim=1)[:,None,None,:]
    with torch.inference_mode():
        compressed=F.scaled_dot_product_attention(
            q.float(),km.float(),vm.float(),attn_mask=allowed,dropout_p=0.0
        )
    pred=truth_project_core(entry,compressed)
    ans=metrics_with_valid(pred,entry["y"],valid)
    ans["memory_keys"]=int(G*3)
    return ans

landmark_probes=[]
for entry in probe_retrieval_cache:
    qkv=manual_teacher_attention(entry)
    landmark_probes.append(baseline_landmark(entry,qkv,block=16))
landmark_probe=mean_metric(landmark_probes)
print("Untrained deterministic 3-per-16 landmark PROBE:",landmark_probe)
print("Diagnostic only. V3.2 learned soft-salience landmarks are different.")


## 2. Does a Gaussian VAE preserve pretrained attention information?

An **AE** learns \(\hat K,\hat V=\operatorname{Dec}(\operatorname{Enc}(K,V))\), penalizing key/value reconstruction error.

A **β-VAE** encodes \(\mu,\log\sigma^2\), samples \(z=\mu+\sigma\odot\epsilon,\ \epsilon\sim\mathcal N(0,I)\), and minimizes

\[
\mathcal L_{\beta\text{-VAE}}=
\underbrace{\mathbb E\big[\|[K,V]-[\hat K,\hat V]\|_2^2\big]}_\text{reconstruction}
+\beta\underbrace{\mathrm{KL}(q(z\mid K,V)\|\mathcal N(0,I)) / d_{KV}}_\text{Gaussian prior regularization}.
\]

In the experiment **all KV representations are RoPE-compatible**: encode *already rotated K* and original V; keep **exact frozen Q**, decode K and V, and run the original full-attention equation. That isolates the information loss caused by a Gaussian latent bottleneck. Deterministic AEs have no KL; denoising AEs add Gaussian noise only while training.

The VAE inference comparison reports both posterior-mean reconstruction and single stochastic sampling. Means are appropriate to measure stable inference; stochastic generation can substantially degrade teacher fidelity. This is **not** a faster attention-replacement method, because it still evaluates full \(QK^\top\). Even a good β-VAE would only justify Gaussian modeling as a complementary representation, not as the solution to token retrieval.

We test **β = 0.0001, 0.001, 0.01, 0.1, 1.0** at latent widths 16 and 32, plus deterministic AE and denoising AE at each width. Optional latent width 64 can be added before running. Comparisons use exactly the same token sample cache and optimizer-update budget per candidate.


In [ ]:
#@title 7. Matched Gaussian bottleneck AE / denoising AE / β-VAE models
class GaussianKVBottleneck(nn.Module):
    def __init__(self,input_dim,latent_dim,kind,beta=0.0,noise_sigma=0.0):
        super().__init__()
        assert kind in ("ae","denoise","vae")
        self.input_dim=input_dim
        self.latent_dim=latent_dim
        self.kind=kind
        self.beta=float(beta)
        self.noise_sigma=float(noise_sigma)
        mid=192
        self.encoder=nn.Sequential(
            nn.LayerNorm(input_dim),nn.Linear(input_dim,mid),nn.GELU()
        )
        self.mu=nn.Linear(mid,latent_dim)
        self.logvar=nn.Linear(mid,latent_dim) if kind=="vae" else None
        self.decoder=nn.Sequential(
            nn.Linear(latent_dim,mid),nn.GELU(),nn.Linear(mid,input_dim)
        )

    def forward(self,x,sample=None):
        training_sample=self.training if sample is None else bool(sample)
        x_in=x
        if self.kind=="denoise" and self.training:
            x_in=x+torch.randn_like(x)*self.noise_sigma
        h=self.encoder(x_in)
        mu=self.mu(h)
        if self.kind=="vae":
            logvar=self.logvar(h).clamp(-10.0,6.0)
            z=mu+torch.exp(.5*logvar)*torch.randn_like(mu) if training_sample else mu
            kl=-.5*(1+logvar-mu.square()-logvar.exp()).sum(-1).mean()
        else:
            logvar=None
            z=mu
            kl=x.new_zeros(())
        decoded=self.decoder(z)
        return decoded,kl,mu,logvar

def variant_configs():
    configs=[]
    for d in LATENT_DIMS:
        configs.append(dict(name=f"ae_z{d}",kind="ae",beta=0.0,dim=d,noise=0.0))
        configs.append(dict(name=f"denoise005_z{d}",kind="denoise",
                            beta=0.0,dim=d,noise=0.05))
        for beta in VAE_BETAS:
            text_beta=str(beta).replace(".","p")
            configs.append(dict(name=f"vae_b{text_beta}_z{d}",
                                kind="vae",beta=float(beta),dim=d,noise=0.0))
    return configs
vae_configs=variant_configs()
print(f"Gaussian bottleneck variants: {len(vae_configs)}")
print("Train models:",[c["name"] for c in vae_configs])

def build_bottleneck(c,dim):
    return GaussianKVBottleneck(dim,c["dim"],c["kind"],
                                beta=c["beta"],noise_sigma=c["noise"]).to(device)


In [ ]:
#@title 8. Extract frozen case-disjoint training K/V tokens; Gaussian checks
# No representation learning from probe/audit labels or teacher-output targets.
# Up to 1024 randomly sampled valid head-token pairs per training cached batch.
train_vectors=[]
heads=int(teacher.encoder.layers[LAYER].attn.num_heads)
head_dim=int(teacher.encoder.layers[LAYER].attn.head_dim)
KV_DIM=head_dim*2

for idx in range(TRAIN_CACHE_BATCHES):
    entry=cache_teacher_io(LAYER,train_items,1,offset=idx*BATCH)[0]
    # Extract exactly the pretrained RoPE-transformed K and original V.
    # Do not build O(T²) teacher attention weights for this training-only step.
    module=teacher.encoder.layers[LAYER].attn
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        x=entry["x"]
        bb,tt,hidden=x.shape
        qkv_full=module.Wqkv(x).view(bb,tt,3,module.num_heads,module.head_dim)
        cos,sin=module.rotary_emb(qkv_full,
            position_ids=entry["call_kwargs"].get("position_ids"))
        _q,_k,_v=qkv_full.transpose(3,1).unbind(dim=2)
        _q,_k=apply_rotary_pos_emb(_q,_k,cos,sin)
    kv=torch.cat((_k,_v),dim=-1)
    # Flatten [B,H,T,2d] -> [B,H*T,2d], keep only valid tokens.
    B,H,T,D=kv.shape
    flat=kv.permute(0,2,1,3).reshape(B*T*H,D).float()
    allowed=entry["valid"][:,:,None].expand(B,T,H).reshape(-1)
    vec=flat[allowed]
    sample_len=min(len(vec),VAE_TRAIN_SAMPLE_PER_BATCH)
    order=torch.randperm(len(vec),device=vec.device)[:sample_len]
    train_vectors.append(vec.index_select(0,order).cpu())
    del entry,qkv_full,_q,_k,_v,cos,sin,vec,flat,kv
    if (idx+1)%16==0: print("Training K/V extraction:",idx+1,"/",TRAIN_CACHE_BATCHES)
X_train=torch.cat(train_vectors,dim=0).float()
del train_vectors
assert X_train.shape[1]==KV_DIM and torch.isfinite(X_train).all()
print("Actual K/V train row matrix:",tuple(X_train.shape),
      "| key dim:",head_dim,"heads:",heads)
assert len(X_train)>VAE_BATCH
# Perturbations are independent of train data; test forward/backward before loops.
c=next(c for c in vae_configs if c["kind"]=="vae")
m=build_bottleneck(c,KV_DIM)
x=X_train[:min(128,len(X_train))].to(device)
out,kl,mu,logvar=m(x,sample=True)
loss=(out-x).square().mean()+c["beta"]*kl/KV_DIM
loss.backward()
assert torch.isfinite(loss) and bool(torch.isfinite(m.mu.weight.grad).all())
print("β-VAE reparameterization/gradient preflight: PASS",
      "| reconstruction MSE:",float(((out-x).square().mean()).detach()),
      "| KL per input dimension:",float((kl/KV_DIM).detach()))
del m,out,kl,mu,logvar,loss,x
torch.cuda.empty_cache()


In [ ]:
#@title 9. Fit all AE / denoising AE / Gaussian β-VAE variants with same budget
def bottleneck_state_cpu(model):
    return {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

def train_gaussian_variant(c):
    model=build_bottleneck(c,KV_DIM)
    ckpt=OUTPUT_DIR/(c["name"]+".pt")
    if ckpt.exists():
        info=torch.load(ckpt,map_location="cpu",weights_only=False)
        if info["config"]!=c or info["steps"]!=VAE_STEPS:
            raise RuntimeError(f"Checkpoint mismatch: {ckpt}; choose new OUTPUT_DIR")
        model.load_state_dict(info["state_dict"])
        model.eval()
        print("Resumed",c["name"])
        return model,info["history"],str(ckpt)
    gen=torch.Generator().manual_seed(SEED + 23*vae_configs.index(c))
    model.train()
    optimizer=torch.optim.AdamW(model.parameters(),lr=0.001,weight_decay=1e-5)
    history=[]
    for step in range(VAE_STEPS):
        row_idx=torch.randint(len(X_train),(VAE_BATCH,),generator=gen)
        selected=X_train.index_select(0,row_idx).to(device)
        optimizer.zero_grad(set_to_none=True)
        decoded,kl,_,_=model(selected,sample=(c["kind"]=="vae"))
        reconstruction=(decoded-selected).square().mean()
        # Normalize total KL by KV_DIM so beta values share scale.
        objective=reconstruction+c["beta"]*kl/KV_DIM
        if not bool(torch.isfinite(objective)):
            raise RuntimeError(f"Non-finite loss for {c['name']} at step {step}")
        objective.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if (step+1)%40==0 or step==0 or step+1==VAE_STEPS:
            history.append(dict(step=step+1,reconstruction=float(reconstruction.detach()),
                                kl_per_dim=float((kl/KV_DIM).detach()),
                                loss=float(objective.detach())))
    model.eval()
    torch.save(dict(version="3.3",config=c,steps=VAE_STEPS,
                    state_dict=bottleneck_state_cpu(model),history=history),ckpt)
    print(f"{c['name']:<23} final training MSE={history[-1]['reconstruction']:.5f} "
          f"KL/d={history[-1]['kl_per_dim']:.5f}")
    return model,history,str(ckpt)

vae_models={}
vae_train={}
for config in vae_configs:
    model,history,ckpt=train_gaussian_variant(config)
    vae_models[config["name"]]=model
    vae_train[config["name"]]=dict(config=config,history=history,checkpoint=ckpt)
print("All Gaussian bottlenecks trained/resumed. Official test untouched.")


In [ ]:
#@title 10. Probe: actual teacher-attention reconstruction through Gaussian bottlenecks
# Baseline is original teacher Q with no K/V compression, not an attention-free model.
def reconstruct_kv(qkv,model,sample=False):
    key,val=qkv["k"],qkv["v"]
    b,h,t,d=key.shape
    source=torch.cat((key,val),dim=-1).float()
    with torch.no_grad():
        decoded,kl,mu,logvar=model(source.reshape(-1,2*d),sample=sample)
    kd,vd=decoded.view(b,h,t,2*d).split(d,dim=-1)
    return kd,vd,mu,logvar

def score_gaussian_replacement(entry,qkv,model,sample=False):
    module=teacher.encoder.layers[LAYER].attn
    with torch.inference_mode():
        kh,vh,mu,logvar=reconstruct_kv(qkv,model,sample=sample)
        q=qkv["q"].float()
        attn_mask=entry["call_kwargs"].get("attention_mask")
        # PyTorch SDPA requires float additive masks to match FP32 Q/K/V.
        if attn_mask is not None and attn_mask.dtype != torch.bool:
            attn_mask=attn_mask.float()
        core=F.scaled_dot_product_attention(
            q,kh,vh,attn_mask=attn_mask,dropout_p=0.0
        )
        pred=truth_project_core(entry,core)
        score=metrics_with_valid(pred,entry["y"],qkv["valid"])
        key=qkv["k"].float()
        value=qkv["v"].float()
        mask=qkv["valid"][:,None,:,None]
        score["key_nmse"]=float((((kh-key)*mask).square().sum())/
                                ((key*mask).square().sum()).clamp_min(1e-8))
        score["value_nmse"]=float((((vh-value)*mask).square().sum())/
                                  ((value*mask).square().sum()).clamp_min(1e-8))
        # Aggregated-posterior statistics (not proof of multivariate normality).
        mu=mu.reshape(key.shape[0],key.shape[1],key.shape[2],-1)
        mup=mu[qkv["valid"][:,None,:].expand(key.shape[:3])]
        score["latent_mean_abs"]=float(mup.mean(0).abs().mean())
        score["latent_var_mean"]=float(mup.var(0,unbiased=False).mean())
        if logvar is not None:
            lv=logvar.reshape(mu.shape)[qkv["valid"][:,None,:].expand(key.shape[:3])]
            score["posterior_variance_mean"]=float(lv.exp().mean())
            score["posterior_kl_per_latent_dim"]=float(
                (-0.5*(1+lv-mup.square()-lv.exp())).sum(-1).mean()/mu.shape[-1]
            )
            # Sample from q(z|K,V) to probe aggregated-latent Gaussianity.
            zstat=mup+(.5*lv).exp()*torch.randn_like(mup)
        else:
            score["posterior_variance_mean"]=0.0
            score["posterior_kl_per_latent_dim"]=0.0
            zstat=mup
        zmean=zstat.mean(0)
        zvar=zstat.var(0,unbiased=False)
        centered=(zstat-zmean)/zvar.clamp_min(1e-8).sqrt()
        score["aggregate_mean_abs"]=float(zmean.abs().mean())
        score["aggregate_variance_error"]=float((zvar-1).abs().mean())
        score["aggregate_skew_abs"]=float(centered.pow(3).mean(0).abs().mean())
        score["aggregate_excess_kurtosis_abs"]=float(
            (centered.pow(4).mean(0)-3).abs().mean()
        )
    return score

probe_kv_entries=cache_teacher_io(LAYER,probe_items,PROBE_CACHE_BATCHES)
probe_gaussian={}
for c in vae_configs:
    model=vae_models[c["name"]]
    vals=[]
    for entry in probe_kv_entries:
        qkv=manual_teacher_attention(entry)
        vals.append(score_gaussian_replacement(entry,qkv,model,sample=False))
        del qkv
    summary=mean_metric(vals)
    selection=summary["nmse"]+1.1*(1-summary["cosine"])
    probe_gaussian[c["name"]]=dict(metrics=summary,score=selection,config=c)
    print(f"{c['name']:<24} probe cos={summary['cosine']:.5f} "
          f"NMSE={summary['nmse']:.5f} keyNMSE={summary['key_nmse']:.4f} "
          f"valueNMSE={summary['value_nmse']:.4f} KL/z={summary['posterior_kl_per_latent_dim']:.4f}")
# Selection is probe-only and declared before opening the audit cache.
ranked_vae=sorted(probe_gaussian,key=lambda key:probe_gaussian[key]["score"])
best_ae=next(name for name in ranked_vae if vae_train[name]["config"]["kind"]=="ae")
best_vae=next(name for name in ranked_vae if vae_train[name]["config"]["kind"]=="vae")
best_denoise=next(name for name in ranked_vae if vae_train[name]["config"]["kind"]=="denoise")
print("Probe selected: AE=",best_ae," VAE=",best_vae," Denoise=",best_denoise)
# Gaussian noise sensitivity for the probe-selected VAE; no retraining.
stochastic_probe=[]
for entry in probe_kv_entries:
    qkv=manual_teacher_attention(entry)
    stochastic_probe.append(score_gaussian_replacement(
        entry,qkv,vae_models[best_vae],sample=True))
stochastic_probe=mean_metric(stochastic_probe)
print("Best VAE stochastic PROBE:",stochastic_probe)


In [ ]:
#@title 11. One-pass held-out audit for exact oracle and Gaussian variants
audit_entries=cache_teacher_io(LAYER,audit_items,AUDIT_CACHE_BATCHES)
audit_retrieval=[]
audit_landmarks=[]
for i,entry in enumerate(audit_entries[:RETRIEVAL_BATCHES]):
    qkv=manual_teacher_attention(entry)
    audit_retrieval.append(oracle_batch(entry,qkv))
    audit_landmarks.append(baseline_landmark(entry,qkv,block=16))
    print(f"Audit exact oracle batch {i+1}/{RETRIEVAL_BATCHES}")
audit_oracles={name:mean_metric([r[name] for r in audit_retrieval]) for name in ORACLE_NAMES}
audit_landmark=mean_metric(audit_landmarks)
print("AUDIT retrieval:")
for name,r in audit_oracles.items():
    print(f" {name:<29} cosine={r['cosine']:.5f} NMSE={r['nmse']:.5f} mass={r['captured_attention_mass']:.3f}")
print("AUDIT deterministic 3/16 landmarks:",audit_landmark)

names_audit=([c["name"] for c in vae_configs] if AUDIT_ALL_VAEs else
             list(dict.fromkeys([best_ae,best_denoise,best_vae])))
audit_gaussian={}
for name in names_audit:
    vals=[]
    for entry in audit_entries:
        qkv=manual_teacher_attention(entry)
        vals.append(score_gaussian_replacement(entry,qkv,vae_models[name],sample=False))
    audit_gaussian[name]=mean_metric(vals)
    print(f"AUDIT Gaussian {name:<24} cosine={audit_gaussian[name]['cosine']:.5f} "
          f"NMSE={audit_gaussian[name]['nmse']:.5f}")
print("Audit is for reporting; winner remains fixed from probe metrics.")


In [ ]:
#@title 12. Probe-selected AE and VAE: actual layer-15 end-to-end and latency
# Diagnostic only: still uses FULL Q-K SDPA after KV decode; expected to be slower.
class KVDecodedAttention(nn.Module):
    def __init__(self,orig,bottleneck):
        super().__init__()
        self.config=orig.config
        self.num_heads=orig.num_heads
        self.head_dim=orig.head_dim
        self.hidden_size=int(orig.config.hidden_size)
        self.Wqkv=copy.deepcopy(orig.Wqkv).requires_grad_(False)
        self.Wo=copy.deepcopy(orig.Wo).requires_grad_(False)
        self.rotary_emb=copy.deepcopy(orig.rotary_emb).requires_grad_(False)
        self.out_drop=copy.deepcopy(orig.out_drop).requires_grad_(False)
        self.bottleneck=copy.deepcopy(bottleneck).requires_grad_(False)
        self.eval()

    def forward(self,hidden_states,attention_mask=None,position_ids=None,**kwargs):
        B,T,D=hidden_states.shape
        h,d=self.num_heads,self.head_dim
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            qkv=self.Wqkv(hidden_states).view(B,T,3,h,d)
            if position_ids is None:
                position_ids=torch.arange(T,device=hidden_states.device)[None,:]
            cos,sin=self.rotary_emb(qkv,position_ids=position_ids)
            q,k,v=qkv.transpose(3,1).unbind(dim=2)
            q,k=apply_rotary_pos_emb(q,k,cos,sin)
        kv=torch.cat((k.float(),v.float()),dim=-1)
        decoded,_,_,_=self.bottleneck(kv.reshape(-1,2*d),sample=False)
        k_hat,v_hat=decoded.view(B,h,T,2*d).split(d,dim=-1)
        mask32=attention_mask
        if mask32 is not None and mask32.dtype != torch.bool:
            mask32=mask32.float()
        core=F.scaled_dot_product_attention(
            q.float(),k_hat,v_hat,attn_mask=mask32,dropout_p=0.0
        )
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            out=self.Wo(core.transpose(1,2).contiguous().reshape(B,T,D))
        return self.out_drop(out),None

def compare_student(model,items):
    n=agree=teacher_gold=student_gold=0
    divergences=[]
    for off in range(0,len(items),BATCH):
        selected=items[off:off+BATCH]
        b=direct_batch(selected)
        with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
            teacher_logits,_=teacher(**b)
            student_logits,_=model(**b)
        valid=b["marker_mask"].bool()
        tp=torch.softmax(teacher_logits.float().masked_fill(~valid,-1e9),dim=-1)
        sp=torch.softmax(student_logits.float().masked_fill(~valid,-1e9),dim=-1)
        ti=tp.argmax(-1)
        si=sp.argmax(-1)
        gold=torch.tensor([it["gold_index"] for it in selected],device=device)
        n+=len(selected)
        agree+=int((ti==si).sum())
        teacher_gold+=int((ti==gold).sum())
        student_gold+=int((si==gold).sum())
        m=0.5*(tp+sp)
        js=0.5*((tp*(tp.clamp_min(1e-8)/m.clamp_min(1e-8)).log()).sum(-1)
                +(sp*(sp.clamp_min(1e-8)/m.clamp_min(1e-8)).log()).sum(-1))
        divergences.extend(js.cpu().tolist())
    return dict(decisions=n,agreement=agree/max(n,1),
                teacher_gold_accuracy=teacher_gold/max(n,1),
                student_gold_accuracy=student_gold/max(n,1),
                gold_accuracy_drop=(teacher_gold-student_gold)/max(n,1),
                mean_js=float(np.mean(divergences)))

def gpu_latency_ms(fn,warmup=5,repeats=15):
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        for _ in range(warmup): fn()
        torch.cuda.synchronize()
        ticks=[]
        for _ in range(repeats):
            t0=time.perf_counter()
            fn()
            torch.cuda.synchronize()
            ticks.append((time.perf_counter()-t0)*1e3)
    return float(np.median(ticks))

teacher_attn=teacher.encoder.layers[LAYER].attn
trial=audit_entries[0]
sample_input=audit_items[:BATCH]
b=direct_batch(sample_input)
audit_e2e={}
for name in list(dict.fromkeys([best_ae,best_vae,best_denoise])):
    repl=KVDecodedAttention(teacher_attn,vae_models[name]).to(device).eval()
    prior=student_eval.encoder.layers[LAYER].attn
    try:
        student_eval.encoder.layers[LAYER].attn=repl
        quality=compare_student(student_eval,audit_items)
        original_ms=gpu_latency_ms(lambda: teacher_attn(
            *trial["call_args"],**trial["call_kwargs"]
        ))
        student_ms=gpu_latency_ms(lambda: repl(
            trial["x"],attention_mask=trial["mask"],
            position_ids=trial["call_kwargs"].get("position_ids")
        ))
        original_model_ms=gpu_latency_ms(lambda:teacher(**b),warmup=2,repeats=8)
        student_model_ms=gpu_latency_ms(lambda:student_eval(**b),warmup=2,repeats=8)
        audit_e2e[name]=dict(
            metrics=quality,
            layer_teacher_ms=original_ms,
            layer_vae_ms=student_ms,
            layer_speedup=original_ms/max(student_ms,1e-8),
            whole_teacher_ms=original_model_ms,
            whole_vae_ms=student_model_ms,
            whole_speedup=original_model_ms/max(student_model_ms,1e-8)
        )
        print("E2E",name,audit_e2e[name])
    finally:
        student_eval.encoder.layers[LAYER].attn=prior
        student_eval.eval().requires_grad_(False)
    del repl
    torch.cuda.empty_cache()
print("All VAE-KV-decode models retain full attention and are not eligible "
      "as attention-free speedup claims.")


In [ ]:
#@title 13. Save complete V3.3 mathematical retrieval + Gaussian latent report
report=dict(
    version="3.3-retrieval-oracle-gaussian-VAE",
    model=SOURCE_MODEL,layer=LAYER,official_test_used=False,
    train_cases=len(train_rows),probe_cases=len(probe_rows),
    audit_cases=len(audit_rows),
    train_cached_decisions=TRAIN_CACHE_BATCHES*BATCH,
    retrieval_probe_batches=len(probe_retrieval_cache),
    retrieval_audit_batches=len(audit_retrieval),
    window_size=256,window_left=127,window_right=128,
    oracle_probe=probe_oracles,oracle_audit=audit_oracles,
    native_teacher_selectivity_probe=teacher_selectivity,
    nonlearned_landmark_probe=landmark_probe,
    nonlearned_landmark_audit=audit_landmark,
    gaussian_variants=vae_configs,
    gaussian_steps_per_variant=VAE_STEPS,
    gaussian_train_token_head_rows=int(len(X_train)),
    gaussian_train_history=vae_train,
    gaussian_probe=probe_gaussian,
    gaussian_audit=audit_gaussian,
    selected_by_probe=dict(ae=best_ae,vae=best_vae,denoise=best_denoise),
    selected_vae_stochastic_probe=stochastic_probe,
    selected_gaussian_e2e=audit_e2e,
    caveats=[
        "Exact top-k retrieval uses full teacher attention weights and is not an efficient replacement.",
        "Top-k captured mass measures teacher concentration, not end-to-end decision equivalence.",
        "256 local window has 127 left and 128 right key positions when present.",
        "3/16 baseline is deterministic and does not reproduce trained V3.2 landmarks.",
        "VAEs train ONLY on training-split RoPE K and V head-token vectors.",
        "Gaussian KL encourages but does not guarantee aggregate Gaussian posterior.",
        "KV-decoded models STILL USE FULL native Q/K attention: NOT attention-free or faster by construction.",
        "Only the training split of typed-decisions is used; official test remains untouched.",
        "Probe-selected winners are fixed before audit; audit used for reporting.",
    ]
)
outpath=OUTPUT_DIR/"v33_retrieval_gaussian_vae_audit.json"
outpath.write_text(json.dumps(report,indent=2,allow_nan=False),encoding="utf-8")
print("Saved",outpath)
print("\nRETRIEVAL ORACLES — matched held-out audit")
for name,val in audit_oracles.items():
    print(f"{name:<31} cosine={val['cosine']:.4f} mass={val['captured_attention_mass']:.3f}")
print("\nGAUSSIAN — probe-selected best AE / denoise / VAE")
for name in [best_ae,best_denoise,best_vae]:
    print(name,"probe cosine",probe_gaussian[name]["metrics"]["cosine"],
          "audit cosine",audit_gaussian[name]["cosine"])
print("\nGO/STOP:")
if audit_oracles["top_32"]["cosine"]>=.95:
    print("Oracle top-32 retained teacher output well. Next: build efficient query-conditioned sparse retrieval.")
else:
    print("Oracle top-32 below 0.95; richer retrieval and/or more keys likely needed.")
if audit_gaussian[best_vae]["cosine"]>audit_gaussian[best_ae]["cosine"]:
    print("Gaussian β-VAE helped this sampled KV reconstruction metric, but check speed/decision gates.")
else:
    print("Gaussian VAE did not outperform deterministic AE on audited output cosine.")
print("No experiment here certifies attention-free replacement. Do not publish claims from oracle timing.")

import matplotlib.pyplot as plt
x=[8,16,32,64]
y=[audit_oracles[f"top_{k}"]["cosine"] for k in x]
plt.plot(x,y,marker="o",label="Oracle teacher-selected top-K")
plt.axhline(.95,linestyle="--",label="Target cosine 0.95")
plt.axhline(audit_oracles["local_256"]["cosine"],linestyle=":",label="Native local-256")
plt.xlabel("Oracle top-K remote/global keys per query")
plt.ylabel("Layer-15 output cosine")
plt.title("Selective retrieval feasibility (not inference speed)")
plt.grid(alpha=0.3);plt.legend();plt.show()
